In [1]:
import pysam
import pandas as pd
from collections import defaultdict

In [2]:
bam_path = "/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.mismatch1.sorted.bam"
bamfile = pysam.AlignmentFile(bam_path, "rb")

read_stats = defaultdict(lambda: {"NM0": 0, "NM1": 0})

for read in bamfile.fetch(until_eof=True):
    if read.is_unmapped:
        continue

    read_name = read.query_name
    nm_tag = read.get_tag("NM") if read.has_tag("NM") else None

    if nm_tag is not None:
        if nm_tag == 0:
            read_stats[read_name]["NM0"] += 1  
        elif nm_tag == 1:
            read_stats[read_name]["NM1"] += 1

df = pd.DataFrame.from_dict(read_stats, orient="index").reset_index()
df.columns = ["id", "Alignments_NM0", "Alignments_NM1"]
df["Alignments_NM_less_than_1"] = df["Alignments_NM0"] + df["Alignments_NM1"]

df.to_csv("/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.processed.tsv", sep="\t", index=False)